# Strong Baseline MLP
This notebook trains a deeper MLP with the Adam optimizer to establish the upper performance limit on the 5,000-sample HIGGS dataset.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from utils.data_utils import load_higgs

def train_and_eval(n_features, use_pca=False):
    X_train, X_val, X_test, y_train, y_val, y_test = load_higgs(
        path="../data/HIGGS.csv.gz",
        n_samples=5000,
        n_features=n_features,
        use_pca=use_pca,
        random_state=42
    )
    
    model = MLPClassifier(
        hidden_layer_sizes=(100, 50, 25),
        activation='relu',
        solver='adam',
        max_iter=500,
        batch_size=32,
        early_stopping=True,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    test_probs = model.predict_proba(X_test)[:, 1]
    test_preds = model.predict(X_test)
    
    # Convert -1/1 labels to 0/1 for AUC
    y_test_01 = (y_test == 1).astype(int)
    
    acc = accuracy_score(y_test, test_preds)
    auc = roc_auc_score(y_test_01, test_probs)
    
    return acc, auc

print("--- 2 Features Strong Baseline ---")
acc_2, auc_2 = train_and_eval(n_features=2)
print(f"Test Acc: {acc_2:.4f}, Test AUC: {auc_2:.4f}")

print("\n--- 28 Features Strong Baseline ---")
acc_28, auc_28 = train_and_eval(n_features=28)
print(f"Test Acc: {acc_28:.4f}, Test AUC: {auc_28:.4f}")